# 10 — Fine-tuning: Zoobot ConvNeXt-Nano (Local — RTX 5060 Ti 16 GB)

Fine-tuning de **Zoobot (ConvNeXt-Nano)** preentrenado con **92M+ anotaciones de Galaxy Zoo** para clasificación morfológica de galaxias (6 clases).  
Versión local optimizada para **NVIDIA RTX 5060 Ti 16 GB** (Blackwell, sm_120).

| Hiperparámetro | Valor |
|---|---|
| Batch size | 32 (16 GB VRAM) — reducir a 16 en 8 GB |
| Learning rate | 1e-4 (recomendado por Zoobot docs) |
| Layer decay | 0.75 (default Zoobot — reduce LR en capas profundas) |
| Weight decay | 0.05 (default Zoobot — AdamW L2) |
| Head dropout | 0.5 (default Zoobot) |
| Optimizer | AdamW (gestionado por Zoobot/timm internamente) |
| Scheduler | Cosine (via `scheduler_kwargs` de Zoobot) |
| Epochs | 50 (+ early stopping, paciencia=10) |
| AMP | ✅ `precision='16-mixed'` (Lightning nativo) |
| Training mode | `'full'` (backbone + head) |
| IMAGE_SIZE | 224 px (resolución nativa Zoobot) |

> **¿Por qué Zoobot?**  
> Zoobot es un framework especializado en clasificación de morfología de galaxias, entrenado con >92 millones de respuestas de voluntarios en Galaxy Zoo (DECaLS, GZ2, Hubble, CANDELS, etc.).  
> A diferencia de modelos preentrenados en ImageNet (gatos, perros, autos), **Zoobot ya entiende la estructura de galaxias**: brazos espirales, barras, bordes, formas elípticas e irregulares.  
> Esto le da una **ventaja masiva** para fine-tuning en clasificación morfológica.

> **¿Por qué ConvNeXt-Nano?**  
> ConvNeXt-Nano es la arquitectura más ligera de Zoobot (~15M params), ideal para GPU de 16 GB.  
> Alternativas más grandes (`convnext_small`, `maxvit_small`) requieren más VRAM pero pueden dar mejor precisión.

> **Modo Full Fine-tuning (`training_mode='full'`):**  
> Entrenamos **todas las capas** (encoder + cabeza).  
> Zoobot aplica **layer decay** automáticamente: las capas más profundas reciben LR más bajo, preservando las features astronómicas aprendidas.

> **⚠️ IMPORTANTE — Zoobot es un LightningModule:**  
> A diferencia de nuestros otros notebooks (04-08) que usan PyTorch puro, Zoobot está construido sobre **PyTorch Lightning**.  
> El entrenamiento usa `trainer.fit()` nativo de Lightning, lo cual gestiona automáticamente:  
> optimizer (AdamW con layer decay), AMP, checkpoints, early stopping, y logging.  
> Usamos callbacks personalizados para mantener compatibilidad con nuestro formato CSV de logs.

## Sección 0 — Instalación de dependencias

Ejecuta esta celda **una sola vez** en el entorno nuevo. Reinicia el kernel después si es la primera instalación.

**Zoobot** requiere: `lightning>=2.1`, `timm>=0.9.2`, `albumentations`, `galaxy-datasets`, `torchmetrics`.

In [ ]:
import subprocess, sys

# PyTorch con CUDA 12.8 (necesario para RTX 5060 Ti / Blackwell sm_120)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
    '--quiet',
], check=True)

# Instalar Zoobot con soporte PyTorch
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'zoobot[pytorch]',
], check=True)

# Dependencias adicionales para evaluación y plots
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'Pillow', 'scikit-learn', 'tqdm',
], check=True)

print('Instalación completada. Reinicia el kernel si es la primera vez.')

## Sección 1 — Imports

In [ ]:
import gc
import os
import sys
import time
import pathlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Lightning (Zoobot 2.0 usa 'lightning', no 'pytorch_lightning')
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, Callback
from lightning.pytorch.callbacks import LearningRateMonitor

# Zoobot
from zoobot.pytorch.training import finetune

# Métricas
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print(f'PyTorch   : {torch.__version__}')
print(f'Lightning : {L.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute   : sm_{cap[0]}{cap[1]}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device    : {device}')

try:
    import zoobot
    print(f'Zoobot    : {zoobot.__version__}')
except (AttributeError, ImportError):
    print('Zoobot    : instalado (versión no disponible via __version__)')

## Sección 2 — Configuración

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Paths locales (relativo a la raíz del repo)
# ──────────────────────────────────────────────────────────────────────────────
_LOCAL     = pathlib.Path('../data')
IMAGES_DIR = _LOCAL / 'images_gz2' / 'images'
SPLITS_DIR = _LOCAL / 'splits'
CKPT_DIR   = pathlib.Path('../models/checkpoints/zoobot_convnext_base')
LOG_DIR    = pathlib.Path('../logs')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Verificar que los datos existen
assert IMAGES_DIR.exists(), f'No se encontró IMAGES_DIR: {IMAGES_DIR.resolve()}'
assert (SPLITS_DIR / 'train.csv').exists(), f'No se encontró train.csv en {SPLITS_DIR.resolve()}'

# ──────────────────────────────────────────────────────────────────────────────
# Modelo Zoobot
# ──────────────────────────────────────────────────────────────────────────────
MODEL_NAME   = 'zoobot_convnext_base'

# Encoder preentrenado desde HuggingFace Hub
# Alternativas (cambiar según VRAM disponible):
#   'hf_hub:mwalmsley/zoobot-encoder-convnext_small'  (~50M params)
#   'hf_hub:mwalmsley/zoobot-encoder-convnext_base'   (~89M params)
#   'hf_hub:mwalmsley/zoobot-encoder-efficientnet_v2_s'
#   'hf_hub:mwalmsley/zoobot-encoder-maxvit_small'
ENCODER_NAME = 'hf_hub:mwalmsley/zoobot-encoder-convnext_base'

NUM_CLASSES  = 6
CLASS_ORDER  = ['Elliptical', 'Lenticular', 'Spiral', 'Barred_Spiral', 'Edge_on', 'Irregular']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# ──────────────────────────────────────────────────────────────────────────────
# Training — configuración siguiendo los defaults de Zoobot docs
# Ref: https://zoobot.readthedocs.io/en/latest/guides/choosing_parameters.html
# ──────────────────────────────────────────────────────────────────────────────
EPOCHS         = 50
EARLY_STOP_PAT = 10
BATCH_SIZE     = 32
NUM_WORKERS    = 0 if os.name == 'nt' else 4

# Zoobot defaults (documentación oficial):
LEARNING_RATE    = 1e-4    # docs: "1e-4 is a good starting point"
LAYER_DECAY      = 0.75   # docs: "default 0.75 — reduce LR en capas profundas"
WEIGHT_DECAY     = 0.05   # docs: "default 0.05 — AdamW L2"
HEAD_DROPOUT     = 0.5    # docs: "default 0.5"
TRAINING_MODE    = 'full' # docs: "'full' = encoder + head, 'head_only' = solo cabeza"

# Imagen
IMAGE_SIZE    = 224       # resolución nativa de los encoders Zoobot
CROP_SIZE     = 320       # CenterCrop para eliminar bordes negros de GZ2 (424×424)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RANDOM_SEED   = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
L.seed_everything(RANDOM_SEED, workers=True)

print(f'IMAGES_DIR     : {IMAGES_DIR.resolve()}')
print(f'SPLITS_DIR     : {SPLITS_DIR.resolve()}')
print(f'CKPT_DIR       : {CKPT_DIR.resolve()}')
print(f'MODEL          : {MODEL_NAME}')
print(f'ENCODER        : {ENCODER_NAME}')
print(f'TRAINING_MODE  : {TRAINING_MODE}')
print(f'EPOCHS         : {EPOCHS}  (early stop paciencia={EARLY_STOP_PAT})')
print(f'BATCH_SIZE     : {BATCH_SIZE}')
print(f'LEARNING_RATE  : {LEARNING_RATE}')
print(f'LAYER_DECAY    : {LAYER_DECAY}')
print(f'WEIGHT_DECAY   : {WEIGHT_DECAY}')
print(f'HEAD_DROPOUT   : {HEAD_DROPOUT}')

## Sección 3 — Pipeline de datos

In [ ]:
# Transforms — mismos que los otros notebooks del pipeline
# Los encoders Zoobot fueron preentrenados con normalización ImageNet.
train_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class GalaxyDataset(Dataset):
    """Dataset idéntico a notebooks 04-08 para compatibilidad."""
    def __init__(self, csv_path, images_dir, transform, class_to_idx):
        df = pd.read_csv(
            csv_path,
            usecols=['img_filename', 'morph_label'],
            dtype={'img_filename': 'str', 'morph_label': 'str'},
        )
        self.filenames = df['img_filename'].to_numpy()
        self.labels    = np.array(
            [class_to_idx[lbl] for lbl in df['morph_label']], dtype=np.int64
        )
        del df
        gc.collect()
        self.images_dir = pathlib.Path(images_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.images_dir / self.filenames[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


# DataLoaders
g = torch.Generator().manual_seed(RANDOM_SEED)

train_dataset = GalaxyDataset(SPLITS_DIR / 'train.csv', IMAGES_DIR, train_transforms, CLASS_TO_IDX)
val_dataset   = GalaxyDataset(SPLITS_DIR / 'val.csv',   IMAGES_DIR, eval_transforms,  CLASS_TO_IDX)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'), generator=g,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

print(f'Train batches : {len(train_loader):,}  ({len(train_dataset):,} imgs)')
print(f'Val   batches : {len(val_loader):,}  ({len(val_dataset):,} imgs)')

In [ ]:
# Class weights para CrossEntropyLoss (compensar desbalance)
_df_w = pd.read_csv(
    SPLITS_DIR / 'train.csv',
    usecols=['morph_label'],
    dtype={'morph_label': 'str'},
)
label_counts = np.bincount(
    _df_w['morph_label'].map(CLASS_TO_IDX).values,
    minlength=NUM_CLASSES,
)
weights      = len(_df_w) / (NUM_CLASSES * label_counts)
class_weights = torch.tensor(weights, dtype=torch.float32)

print('Class weights:')
for cls, w, n in zip(CLASS_ORDER, weights, label_counts):
    print(f'  {cls:<15}  n={n:>6,}   w={w:.4f}')

del _df_w
gc.collect()
print('RAM liberada ✓')

## Sección 4 — Modelo Zoobot

Usamos `FinetuneableZoobotClassifier` de Zoobot para crear el modelo.  
Internamente, esto:
1. Descarga el **encoder ConvNeXt-Nano** preentrenado con >92M anotaciones de Galaxy Zoo desde HuggingFace Hub
2. Agrega una **cabeza de clasificación** (`nn.Linear`) para nuestras 6 clases
3. Configura **AdamW** con layer decay automático y weight decay

**Subclase `WeightedZoobotClassifier`:**  
Zoobot usa `nn.CrossEntropyLoss` sin pesos de clase por defecto.  
Subclasificamos para agregar **class weights** (compensar desbalance) y **métricas F1**.

> **Parámetros clave (de la documentación oficial):**
> - `training_mode='full'` → entrena encoder + cabeza (NO `n_blocks=0`, que es solo cabeza)
> - `layer_decay=0.75` → cada bloque más profundo recibe `lr * 0.75^depth`
> - `weight_decay=0.05` → regularización L2 en AdamW
> - `head_dropout_prob=0.5` → dropout antes de la capa de salida

In [ ]:
class WeightedZoobotClassifier(finetune.FinetuneableZoobotClassifier):
    """
    Subclase de FinetuneableZoobotClassifier que agrega:
    - CrossEntropyLoss con class weights (para desbalance)
    - Logging de F1 macro por epoch (para early stopping y nuestro CSV)
    
    Hereda todo de FinetuneableZoobotClassifier:
    - Encoder preentrenado (timm model)
    - Head con dropout
    - configure_optimizers() con AdamW + layer decay
    - forward() que pasa por encoder → head
    """

    def __init__(self, class_weights_tensor, **kwargs):
        super().__init__(**kwargs)
        # Reemplazar la loss por defecto con weighted CrossEntropyLoss
        self.loss = nn.CrossEntropyLoss(weight=class_weights_tensor)
        self.class_weights_tensor = class_weights_tensor

        # Acumuladores para F1 por epoch
        self._train_preds = []
        self._train_labels = []
        self._val_preds = []
        self._val_labels = []

    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)  # forward: encoder → head
        loss = self.loss(logits, labels)

        # Acumular predicciones para F1
        preds = logits.argmax(dim=1)
        self._train_preds.append(preds.detach().cpu())
        self._train_labels.append(labels.detach().cpu())

        self.log('train/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.loss(logits, labels)

        preds = logits.argmax(dim=1)
        self._val_preds.append(preds.detach().cpu())
        self._val_labels.append(labels.detach().cpu())

        self.log('val/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        if self._train_preds:
            preds = torch.cat(self._train_preds).numpy()
            labels = torch.cat(self._train_labels).numpy()
            f1 = f1_score(labels, preds, average='macro', zero_division=0)
            self.log('train/f1', f1, prog_bar=True)
        self._train_preds.clear()
        self._train_labels.clear()

    def on_validation_epoch_end(self):
        if self._val_preds:
            preds = torch.cat(self._val_preds).numpy()
            labels = torch.cat(self._val_labels).numpy()
            f1 = f1_score(labels, preds, average='macro', zero_division=0)
            self.log('val/f1', f1, prog_bar=True)
        self._val_preds.clear()
        self._val_labels.clear()


# ──────────────────────────────────────────────────────────────────────────────
# Crear el modelo
# ──────────────────────────────────────────────────────────────────────────────
model = WeightedZoobotClassifier(
    class_weights_tensor=class_weights,
    # --- Parámetros de FinetuneableZoobotAbstract ---
    name=ENCODER_NAME,           # encoder preentrenado desde HuggingFace
    num_classes=NUM_CLASSES,      # 6 clases morfológicas
    training_mode=TRAINING_MODE,  # 'full' = encoder + head
    learning_rate=LEARNING_RATE,  # 1e-4 (default docs)
    layer_decay=LAYER_DECAY,      # 0.75 — lr * 0.75^depth por bloque
    weight_decay=WEIGHT_DECAY,    # 0.05 — AdamW L2
    head_dropout_prob=HEAD_DROPOUT,  # 0.5
    # Scheduler: cosine decay nativo de timm
    scheduler_kwargs={
        'sched': 'cosine',
        'num_epochs': EPOCHS,
        'decay_rate': 0.1,
    },
    seed=RANDOM_SEED,
)

# Info del modelo
total_params = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
head_params = sum(p.numel() for p in model.head.parameters())

print(f'Encoder         : {ENCODER_NAME}')
print(f'Training mode   : {TRAINING_MODE}')
print(f'Encoder dim     : {model.encoder_dim}')
print(f'Encoder params  : {encoder_params:,}')
print(f'Head params     : {head_params:,}')
print(f'Total params    : {total_params:,}')
print(f'Layer decay     : {LAYER_DECAY} (lr * {LAYER_DECAY}^depth per block)')

## Sección 5 — Infraestructura de entrenamiento (Lightning)

A diferencia de los notebooks 04-08, aquí NO definimos optimizer/scheduler manualmente.  
Zoobot lo gestiona internamente en `configure_optimizers()` con:
- **AdamW** con layer decay (via `timm.optim.create_optimizer_v2`)
- **Cosine scheduler** (via `timm.scheduler.create_scheduler_v2`)

Definimos **callbacks** de Lightning para:
- `EarlyStopping`: detener si val F1 no mejora
- `ModelCheckpoint`: guardar best/latest
- `CSVLogger`: logging CSV compatible con nuestro pipeline
- `LearningRateMonitor`: registrar LR por epoch

In [ ]:
# Callback personalizado para guardar historial CSV compatible con pipeline
class PipelineCSVLogger(Callback):
    """
    Guarda un CSV con el formato idéntico a los notebooks 04-08:
    epoch,train_loss,train_f1,val_loss,val_f1,lr,elapsed_s,is_best
    """
    def __init__(self, log_path, model_name):
        super().__init__()
        self.log_path = pathlib.Path(log_path)
        self.model_name = model_name
        self.history = []
        self.best_val_f1 = 0.0
        self._epoch_start_time = None

    def on_train_epoch_start(self, trainer, pl_module):
        self._epoch_start_time = time.time()
        ts = datetime.now().strftime('%H:%M:%S')
        epoch = trainer.current_epoch + 1
        print(f'\n[{ts}] ── Epoch {epoch:02d}/{trainer.max_epochs} ───────────────────────────────')

    def on_validation_epoch_end(self, trainer, pl_module):
        # Saltar el sanity check (epoch antes de entrenar)
        if trainer.sanity_checking:
            return

        elapsed = time.time() - self._epoch_start_time if self._epoch_start_time else 0
        epoch = trainer.current_epoch + 1

        # Obtener métricas del callback_metrics de Lightning
        metrics = trainer.callback_metrics
        train_loss = metrics.get('train/loss', torch.tensor(0)).item()
        train_f1   = metrics.get('train/f1', torch.tensor(0)).item()
        val_loss   = metrics.get('val/loss', torch.tensor(0)).item()
        val_f1     = metrics.get('val/f1', torch.tensor(0)).item()

        # LR actual (del primer param group)
        lr = trainer.optimizers[0].param_groups[0]['lr']

        is_best = val_f1 > self.best_val_f1
        if is_best:
            self.best_val_f1 = val_f1

        row = {
            'epoch':      epoch,
            'train_loss': round(train_loss, 6),
            'train_f1':   round(train_f1, 6),
            'val_loss':   round(val_loss, 6),
            'val_f1':     round(val_f1, 6),
            'lr':         round(lr, 8),
            'elapsed_s':  round(elapsed, 1),
            'is_best':    is_best,
        }
        self.history.append(row)

        # Guardar CSV incremental
        pd.DataFrame(self.history).to_csv(self.log_path, index=False)

        # Print resumen
        best_tag  = '  ← BEST' if is_best else ''
        early_tag = '' if is_best else f'  [no mejora: best={self.best_val_f1:.4f}]'
        print(
            f'  train  loss={train_loss:.4f}  F1={train_f1:.4f}\n'
            f'  val    loss={val_loss:.4f}  F1={val_f1:.4f}{best_tag}{early_tag}\n'
            f'  time   {elapsed:.0f}s   LR={lr:.2e}',
            flush=True,
        )


# ──────────────────────────────────────────────────────────────────────────────
# Callbacks
# ──────────────────────────────────────────────────────────────────────────────
LOG_CSV = LOG_DIR / f'{MODEL_NAME}_log.csv'

csv_logger = PipelineCSVLogger(LOG_CSV, MODEL_NAME)

early_stop = EarlyStopping(
    monitor='val/f1',
    patience=EARLY_STOP_PAT,
    mode='max',             # mayor F1 es mejor
    verbose=True,
)

# Un solo ModelCheckpoint — Lightning no permite múltiples con el mismo state_key.
# save_last=True guarda automáticamente 'last.ckpt' para reanudar.
# save_top_k=1 + monitor='val/f1' guarda el mejor modelo como 'best.ckpt'.
checkpoint_cb = ModelCheckpoint(
    dirpath=str(CKPT_DIR),
    filename='best',
    monitor='val/f1',
    mode='max',
    save_top_k=1,
    save_last=True,      # Guarda 'last.ckpt' automáticamente cada epoch
    verbose=True,
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

print('Callbacks configurados:')
print(f'  EarlyStopping  : monitor=val/f1, patience={EARLY_STOP_PAT}, mode=max')
print(f'  ModelCheckpoint: best.ckpt (by val/f1) + last.ckpt (every epoch)')
print(f'  PipelineCSV    : {LOG_CSV}')
print(f'  LRMonitor      : per epoch')

## Sección 6 — Reanudar desde checkpoint

Si existe `last.ckpt` en `CKPT_DIR`, el entrenamiento continúa desde donde se detuvo.  
Para empezar desde cero, borra o renombra `last.ckpt`.

> **Nota:** Lightning gestiona automáticamente la reanudación: restaura modelo, optimizer, scheduler, epoch, y callbacks.

In [ ]:
# Lightning usa 'last.ckpt' con save_last=True
LAST_CKPT = CKPT_DIR / 'last.ckpt'

resume_from = None
if LAST_CKPT.exists():
    resume_from = str(LAST_CKPT)
    print(f'Checkpoint encontrado: {LAST_CKPT}')
    print('El entrenamiento se reanudará desde este checkpoint.')
else:
    print('Sin checkpoint — entrenamiento desde cero')

## Sección 7 — Trainer de Lightning

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Crear el Trainer de Lightning
# ──────────────────────────────────────────────────────────────────────────────
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else '32-true',
    callbacks=[
        early_stop,
        checkpoint_cb,
        csv_logger,
        lr_monitor,
    ],
    default_root_dir=str(CKPT_DIR),
    log_every_n_steps=50,
    enable_progress_bar=True,
    deterministic=False,
)

print(f'Trainer creado:')
print(f'  max_epochs     : {EPOCHS}')
print(f'  accelerator    : {trainer.accelerator.__class__.__name__}')
print(f'  precision      : {trainer.precision}')
print(f'  resume_from    : {resume_from or "None (fresh start)"}')

## Sección 8 — Loop de entrenamiento

In [ ]:
print(f'Iniciando entrenamiento: {MODEL_NAME}')
print(f'Training mode: {TRAINING_MODE} (encoder + head)')
print(f'Early stopping: paciencia = {EARLY_STOP_PAT} epochs sin mejora en val/f1')
print('=' * 72)

t_start = time.time()

# trainer.fit() gestiona TODO:
# - Optimizer (AdamW con layer decay)
# - Scheduler (cosine via timm)
# - AMP (precision='16-mixed')
# - Checkpoints (via callbacks)
# - Early stopping (via callback)
# - Logging (via PipelineCSVLogger callback)
trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path=resume_from,  # None si es desde cero, path si reanudamos
)

t_total = time.time() - t_start

print('\n' + '=' * 72)
print(f'Entrenamiento completado en {t_total/60:.1f} minutos')
print(f'Mejor val F1 = {csv_logger.best_val_f1:.4f}')
print(f'Best checkpoint: {checkpoint_cb.best_model_path}')

## Sección 9 — Curvas de entrenamiento

In [ ]:
hist_df = pd.DataFrame(csv_logger.history)

# Si la historia está vacía (porque se cargó de un checkpoint),
# intentar cargar del CSV.
if hist_df.empty and LOG_CSV.exists():
    hist_df = pd.read_csv(LOG_CSV)

if hist_df.empty:
    print('No hay historial de entrenamiento para graficar.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{MODEL_NAME} — Training History', fontsize=13, fontweight='bold')

    # Loss
    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train')
    axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # F1
    best_row = hist_df.loc[hist_df['val_f1'].idxmax()]
    axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train')
    axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val')
    axes[1].axvline(best_row['epoch'], color='red', linestyle='--', alpha=0.5,
                    label=f'Best={best_row["val_f1"]:.4f} (ep{int(best_row["epoch"])})')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 macro')
    axes[1].set_title('Macro F1 Score'); axes[1].legend(); axes[1].grid(alpha=0.3)

    # LR
    axes[2].semilogy(hist_df['epoch'], hist_df['lr'], label='LR')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate (log)')
    axes[2].set_title('Learning Rate Schedule'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(LOG_DIR / f'{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada en {LOG_DIR}/{MODEL_NAME}_training_curves.png')

## Sección 10 — Evaluación final (mejor modelo sobre test)

In [ ]:
# Cargar el mejor checkpoint
best_ckpt_path = checkpoint_cb.best_model_path

if not best_ckpt_path or not pathlib.Path(best_ckpt_path).exists():
    # Fallback: buscar best.ckpt en CKPT_DIR
    best_ckpt_path = str(CKPT_DIR / 'best.ckpt')

if pathlib.Path(best_ckpt_path).exists():
    print(f'Cargando mejor modelo: {best_ckpt_path}')
    best_model = WeightedZoobotClassifier.load_from_checkpoint(
        best_ckpt_path,
        class_weights_tensor=class_weights,
    )
    print('Mejor modelo cargado ✓')
else:
    print('best.ckpt no encontrado — usando el estado actual del modelo')
    best_model = model

best_model.eval()
best_model = best_model.to(device)

# Test DataLoader (creado aquí, no antes, para no mantener workers durante training)
test_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'test.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)
print(f'Test batches : {len(test_loader):,}  ({len(test_loader.dataset):,} imgs)')

# Predicciones sobre test set
all_preds  = []
all_labels = []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='  Test', unit='batch'):
        imgs = imgs.to(device)
        logits = best_model(imgs)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.tolist())

print('\nClassification Report — Test set:')
print(classification_report(all_labels, all_preds, target_names=CLASS_ORDER, zero_division=0))

test_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
print(f'\nTest F1 macro = {test_f1:.4f}')

In [ ]:
# Confusion matrix normalizada
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (test set)', fontsize=12, fontweight='bold')

for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',     'd'),
    (axes[1], cm_norm, 'Normalized', '.2f'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
        ax=ax, vmin=0, vmax=(1 if fmt == '.2f' else None),
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticklabels(CLASS_ORDER, rotation=30, ha='right')
    ax.set_yticklabels(CLASS_ORDER, rotation=0)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 11 — Resumen

**Artefactos generados:**
- `models/checkpoints/zoobot_convnext_nano/best.ckpt` — mejor checkpoint por val F1 (Lightning)
- `models/checkpoints/zoobot_convnext_nano/last.ckpt` — último checkpoint (para reanudar)
- `models/checkpoints/zoobot_convnext_nano/epoch_XXX.ckpt` — hitos cada 5 epochs
- `logs/zoobot_convnext_nano_log.csv` — historial epoch por epoch
- `logs/zoobot_convnext_nano_training_curves.png`
- `logs/zoobot_convnext_nano_confusion_matrix.png`

**Diferencias clave vs notebooks 04-08 (EfficientNet/ResNet/Swin/MaxViT):**

| Aspecto | Notebooks 04-08 | Este notebook (10) |
|---|---|---|
| Framework | PyTorch puro (manual loop) | PyTorch Lightning (via Zoobot) |
| Preentrenamiento | ImageNet-1K | **92M+ Galaxy Zoo annotations** |
| Optimizer | AdamW manual | AdamW con **layer decay automático** (timm) |
| Scheduler | CosineAnnealingLR | **Cosine via timm** (integrado en Zoobot) |
| AMP | Manual `autocast/GradScaler` | **Lightning `precision='16-mixed'`** |
| Checkpoints | `.pth` manual | **`.ckpt` Lightning** (incluye todo el estado) |
| Class weights | Manual en `CrossEntropyLoss` | Subclase `WeightedZoobotClassifier` |
| Early stopping | Manual counter | **Lightning `EarlyStopping` callback** |
| LR diferencial | 2 param groups | **Layer decay 0.75** (reduce por profundidad) |

**Parámetros de Zoobot (documentación oficial):**
- `training_mode='full'` → entrena encoder + head (≠ `n_blocks=0` que es head only)
- `layer_decay=0.75` → `lr * 0.75^depth` por bloque del encoder
- `weight_decay=0.05` → regularización L2 en AdamW
- `head_dropout_prob=0.5` → dropout antes de la salida
- `learning_rate=1e-4` → "good starting point" según docs

**Para reanudar el entrenamiento:**
Ejecutar el notebook de nuevo sin borrar `last.ckpt`.

**Comparativa acumulada (tras ejecutar 09_evaluation.ipynb):**
- `04` EfficientNet-B3 → best val F1 = 0.6894  /  test F1 = 0.6750  (epoch 16/30)
- `05` ResNet-50       → best val F1 = 0.6914  /  test F1 = 0.6706  (epoch 14/19)
- `06` Swin-S          → best val F1 = 0.6962  /  test F1 = 0.6834  (epoch 25/30)
- `08` MaxViT-T        → best val F1 = 0.6951  /  test F1 = 0.6839  (epoch 17/22)
- `10` **Zoobot ConvNeXt-Nano** → *pendiente de entrenamiento*

**Hipótesis:** Zoobot debería **superar a todos los modelos anteriores** en F1 porque:
1. Su encoder fue preentrenado específicamente en morfología de galaxias (no ImageNet genérico)
2. Ya comprende features como brazos espirales, barras, bordes, irregularidades
3. El full fine-tuning con layer decay adapta todas las capas preservando conocimiento astronómico

**Siguiente paso → notebook de evaluación comparativa final (09_evaluation.ipynb)**

> **Nota sobre checkpoints:**  
> Los checkpoints de Zoobot son `.ckpt` (formato Lightning), no `.pth` (formato PyTorch puro).  
> Para cargar en `09_evaluation.ipynb`, usar:  
> ```python
> model = WeightedZoobotClassifier.load_from_checkpoint('best.ckpt', class_weights_tensor=...)
> ```  
> O extraer el state_dict:  
> ```python
> torch.save(model.state_dict(), 'best.pth')
> ```